# Minimal working example using BERT (`distBERT` model)

In [2]:
# !pip install pandas torch transformers faiss-cpu

## Setup

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

/Users/evanbarnett/Desktop/Northwestern/Classes/de300/data_eng300SP_barnett/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pseudo data creation

In [14]:
# ----------------------------
# 1) Sample pseudo "data"
# ----------------------------
items = pd.DataFrame([
    {"item_id":"i1","title":"Noise Cancelling Headphones","description":"Wireless noise-cancelling headphones with 30-hour battery life","category":"electronics"},
    {"item_id":"i2","title":"Mechanical Keyboard","description":"Mechanical keyboard with RGB and hot-swappable switches","category":"electronics"},
    {"item_id":"i3","title":"Running Shoes","description":"Running shoes designed for long distance comfort and stability","category":"sports"},
    {"item_id":"i4","title":"Vegetarian Cookbook","description":"Cookbook featuring quick vegetarian recipes for busy weeknights","category":"books"},
    {"item_id":"i5","title":"Fitness Smartwatch","description":"Smartwatch with heart rate monitoring, GPS, and sleep tracking","category":"electronics"},
])

events = pd.DataFrame([
    {"user_id":"u1","item_id":"i1","ts":"2026-01-10"},
    {"user_id":"u1","item_id":"i5","ts":"2026-01-11"},
    {"user_id":"u2","item_id":"i2","ts":"2026-01-12"},
    {"user_id":"u3","item_id":"i3","ts":"2026-01-13"},
    {"user_id":"u3","item_id":"i4","ts":"2026-01-14"},
])
events["ts"] = pd.to_datetime(events["ts"])


In [15]:
items_df = pd.read_csv("datasets/bert_data/items.csv")

In [16]:
items_df.head()

,item_id,title,description,category,brand,price,tags
0,i1,Noise Cancelling Headphones,Wireless noise-cancelling headphones with 30-h...,electronics,SoundPeak,199.99,"audio,wireless,travel,premium"
1,i2,Mechanical Keyboard,"Mechanical keyboard with RGB backlighting, hot...",electronics,KeyForge,129.00,"keyboard,gaming,productivity,desk"
2,i3,Running Shoes,Running shoes designed for long distance comfo...,sports,StrideLab,110.00,"running,fitness,outdoors,comfort"
3,i4,Vegetarian Cookbook,"Cookbook featuring quick vegetarian recipes, p...",books,HomeTable Press,24.99,"cooking,vegetarian,recipes,home"
4,i5,Fitness Smartwatch,"Smartwatch with heart rate monitoring, GPS, sl...",electronics,PulsePath,249.00,"wearables,fitness,gps,health"


In [17]:
events_df = pd.read_csv('datasets/bert_data/events.csv')
events_df = events_df[['user_id', 'item_id', 'ts', 'dwell_seconds']]
events_df = events_df[events_df['dwell_seconds'] > 50]

In [18]:
events_df.head()

,user_id,item_id,ts,dwell_seconds
0,u1,i2,2026-01-01 07:47:00,56
2,u1,i2,2026-01-07 14:28:00,72
3,u1,i13,2026-01-07 19:06:00,67
6,u1,i6,2026-01-12 21:40:00,59
7,u1,i17,2026-01-13 14:10:00,99


## BERT encoder (What makes BERT work?)

In [19]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14940.70it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [20]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=128):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [21]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = items["title"] + ". " + items["description"]
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


In [22]:
text_cols = ["title", "description", "category", "brand", "tags", "price"]

items_df["text"] = (
    items_df[text_cols]
    .fillna("")
    .astype(str)
    .apply(
        lambda row: ". ".join(
            f"{col}: {row[col]}" for col in text_cols if row[col].strip()
        ),
        axis=1,
    )
)

item_vecs = bert_embed(items_df["text"].tolist())
item_id_list = items_df["item_id"].tolist()

index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)

## What user-specific data are there?

In [23]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=3):
    hist = (events[events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history"
    text = items.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)


## How to recommend "similar" item with BERT?

In [24]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    user_text, seen = build_user_text(user_id, events, items, N=3)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs


In [25]:

for u in ["u1","u2","u3"]:
    print(u, "->", recommend(u, k=3))


u1 -> ['i17', 'i13', 'i12']
u2 -> ['i9', 'i1', 'i12']
u3 -> ['i6', 'i11', 'i19']


In [26]:
events_df[events_df['user_id'] == 'u5']['item_id'].value_counts()

item_id
i17    3
i9     2
i2     2
i7     2
i14    1
i6     1
i3     1
i19    1
Name: count, dtype: int64

In [28]:
items_df[items_df['item_id'] == 'i17']


,item_id,title,description,category,brand,price,tags,text
16,i17,Smart Home Speaker,Voice-controlled smart speaker with room-filli...,electronics,CasaWave,99.99,"smart-home,audio,assistant,home",title: Smart Home Speaker. description: Voice-...
